# Bioinformatics pipeline testing - sequence inference

This notebook implements DADA2 and VSEARCH _de novo_ pipelines for four different sets of amplicon data (three real and one simulated), covering all steps before taxonomy assignment. It also loads in denoised data from the MetaBEAT pipeline to test alternative taxonomic assignments. The DADA2 workflow is adapted from the [NEOF version](https://github.com/khmaher/HPC_dada2/tree/main?tab=readme-ov-file), and the VSEARCH workflow is based on a published [GitHub pipeline](https://github.com/torognes/vsearch) and [VSEARCH documentation](https://torognes.github.io/vsearch/), both modified for this project.

The pipeline includes:
- Removal of Ns and primers (cutadapt)
- Read quality control, filtering, and trimming (filterAndTrim)
- Error modelling, dereplication, denoising, read merging and chimera removal (DADA2 & VSEARCH)

**NOTE:** There are some places where the test dataset name must be changed to either `Marchamley`, `RingTrial_Sean`, `Windermere_2017` or `Simulated`. Please search and replace.

## 1. Set-up

Install packages, load required functions and get data.

In [0]:
source("Scripts/00_setup.R") #packages and functions

In [0]:
%sh

# needed for packages to install
apt-get -q update

# cutadapt
apt-get -q install -y cutadapt
#pip install -q cutadapt
#cutadapt --version

# vsearch install
apt-get -q install -y vsearch

In [0]:
test_data_name <- "Simulated" # change here to update test data (same name as folder)
test_data_loc <- file.path("Data", "Raw", test_data_name)
results_loc = paste("Results/", test_data_name)
print(head(list.files(test_data_loc)))

In [0]:
manifest <- make_manifest(test_data_loc)

dir.create(
  file.path("Data", "Temp", test_data_name),
  recursive = TRUE,
  showWarnings = FALSE
)

write.csv(
  manifest,
  file.path("Data", "Temp", test_data_name, "manifest.csv"),
  row.names = FALSE
)

head(manifest)

## 2. Removing Ns

In [0]:
source("Scripts/01_remove_Ns.R") # scrapped this step, still creates the objects but we use the original version in the next script

## 3. Identify and remove primers

In [0]:
# Riaz 2011

FWD <- "ACTGGGATTAGATACCCC"
REV <- "TAGAACAGGCTCCTCTAG"

In [0]:
# these would be options in actual pipeline

minimum <- 60 #Minimum read length cutoff. half the length of the amplicon
copies <- 2 #Number of copies of a primer to be removed as sometimes dulication can occur. Recommended minimum is 2

In [0]:
source("Scripts/02_primer_removal.R")

## 4. Generate quality plots

In [0]:
source("Scripts/03_raw_quality_plots.R")

In the plots below:
- Grey-scale heatmap shows the frequency of each quality score along the read lengths - looking for over 30 ish. 
- Green line is median quality score. 
- Orange line are quartiles. 
- Red line at the bottom represent the proportion of reads of that particular length. 

Our reads on average are approx 100bp long after primers removed in the first two samples, which is about right for these primers.

In [0]:
print(plotQualityProfile(fnFs.cut[1:2]))

In [0]:
print(plotQualityProfile(fnRs.cut[1:2]))

## 5. Cleaning the data (filterAndTrim)

Parameters for filterAndTrim include:
- **maxN** - after truncation, sequences with more then X Ns will be disgarded
- **truncQ** - Truncate reads at the first instance of a quality score less then or equal to X
- **rm.phix** - Discard reads that match against the phiX genome
- **maxEE** - After truncation, reads with higher than X expected errors will be discarded
- **minLen** - Remove reads with length less than 60 (note these should gave already been removed by cutadapt)
- **multithread** - input files are filtered in parallel (logical)
- **truncLen** - controls where each read is cut (truncated) based on its length

In [0]:
truncLen=c(80,95) # where the quality dropped in previous plots
maxEE <-  c(2,2) #maxEE value 
truncQ <- 2 #truncQ value
minLen <- 50 #Impose a minimum length cutoff

In [0]:
source("Scripts/04_filterAndTrim.R")

In [0]:
plotQualityProfile(fnFs.filtN[5:6])

In [0]:
plotQualityProfile(fnRs.filtN[1:2])

## 6. Generate error model

In [0]:
source("Scripts/05_generate_error_model.R")

In the plots below:
- Error rates for each possible transition (e.g. A->C, A->G) are shown
- Red line = expected based on quality score
- Black line = estimate 
- Black dots = observed 

We want black lines and black dots to match. We also want to see a rough negative correlation. 

If it looks weird, can increase the number of bases the function is using (default is 100 million).

In [0]:
plotErrors(errF, nominalQ = TRUE)


In [0]:
plotErrors(errR, nominalQ = TRUE)

Common to see 'hook' between 30 - 40 with NovaSeq and MiniSeq data when visualising in DADA2. Couple of forums have discussion on this topic. Not to be of too much concern unless seeing any weird results.

## 7. Deplication, merging and chimera removal

### 7.1. DADA2

In [0]:
source("Scripts/06a_derep_DADA2.R")

### 7.2. VSEARCH de novo

In [0]:
%sh
bash Scripts/06b_derep_VSEARCH.sh Simulated #change to test data

## 8. Sequence tracking & formatting

In [0]:
source("Scripts/07a_sequence_tracking.R")

Input and filtered columns will be the same for both pipelines, as they were done in the filter and trim step. They are not repeated below.

Let's look at just the VSEARCH steps post filterAndTrim. They are in a different order the DADA2 (merging happens first in VSEARCH) so please take note of the column names.

In [0]:
%sh
bash Scripts/07b_sequence_tracking.sh Simulated #change to test data

In [0]:
source("Scripts/07c_format_VSEARCH.R")

The number of sequences should look roughly similar between the two methods.

We now need to get the MetaBEAT data in a similar format to the DADA2 outputs.

In [0]:
%sh
python -u Scripts/07d_format_MetaBEAT.py # parameters to change at top of script

Check out data outputs.

In [0]:
# load dataset
ASV_tab_DADA2 <- read.table(
  file = file.path("Data", "Processed", test_data_name, "06_ASV_counts_DADA2.tsv"),
  header = TRUE,
  sep = "\t",
  row.names = 1,
  check.names = FALSE
)

In [0]:
ASV_tab_VSEARCH <- read.table(
  file = file.path("Data", "Processed", test_data_name, "06b_ASV_counts_VSEARCH.tsv"),
  header = TRUE,
  sep = "\t",
  row.names = 1,
  check.names = FALSE
)

In [0]:
ASV_seq_DADA2 <- readDNAStringSet(file.path("Data", "Processed", test_data_name, "06_ASV_seqs_DADA2.fasta"))
ASV_seq_VSEARCH <- readDNAStringSet(file.path("Data", "Processed", test_data_name, "06b_ASV_seqs_VSEARCH.fasta"))

End of notebook. Please continue to the next notebook to run assign taxonomy.